# Notebook 8.4 - System Tables

Query `system.serving.endpoint_usage`, inspect the request and response fields, and create simple visualizations for endpoint activity.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dbutils.widgets.text("endpoint_name", "mnist-serving-endpoint")
dbutils.widgets.text("lookback_hours", "6")

endpoint_name = dbutils.widgets.get("endpoint_name")
lookback_hours = int(dbutils.widgets.get("lookback_hours"))

usage_schema_df = spark.sql("DESCRIBE TABLE system.serving.endpoint_usage")
served_entities_schema_df = spark.sql("DESCRIBE TABLE system.serving.served_entities")
display(usage_schema_df)
display(served_entities_schema_df)

available_columns = [row.col_name for row in usage_schema_df.collect() if row.col_name and not row.col_name.startswith("#")]
served_entities_columns = [row.col_name for row in served_entities_schema_df.collect() if row.col_name and not row.col_name.startswith("#")]
{
    "endpoint_usage_columns": available_columns,
    "served_entities_columns": served_entities_columns,
}

In [ ]:
timestamp_expr = "eu.request_time"
model_expr = "se.served_entity_name"

query = f"""
SELECT
  se.endpoint_name,
  se.served_entity_name,
  se.entity_name,
  se.entity_type,
  {timestamp_expr} AS timestamp,
  {model_expr} AS model_name_version,
  eu.status_code,
  eu.requester,
  eu.client_request_id,
  eu.databricks_request_id,
  eu.usage_context,
  eu.request_streaming
FROM system.serving.endpoint_usage eu
JOIN system.serving.served_entities se
  ON eu.served_entity_id = se.served_entity_id
WHERE se.endpoint_name = '{endpoint_name}'
  AND {timestamp_expr} >= current_timestamp() - INTERVAL {lookback_hours} HOURS
ORDER BY timestamp DESC
"""

usage_df = spark.sql(query)
display(usage_df)

Expected fields to focus on in this workshop:

- `timestamp`: when the request was served
- `model_name_version`: which served entity handled the call
- `status_code`: request outcome
- `requester`: identity used for the request
- `usage_context`: optional request metadata captured by usage tracking

This system table does not expose full request and response payloads in the schema shown above. It is a usage-tracking table, not a payload log.

For custom model serving endpoints, token and character count columns can legitimately be empty. Do not build the analysis around those fields.

In [ ]:
pdf = usage_df.toPandas()

if pdf.empty:
    print("No endpoint_usage rows found yet. Let 8.3 run for a bit and refresh this notebook.")
else:
    timestamp_col = "timestamp"
    model_col = "model_name_version"
    pdf[timestamp_col] = pd.to_datetime(pdf[timestamp_col])
    pdf[model_col] = pdf[model_col].fillna("unknown")
    pdf["status_code"] = pdf["status_code"].fillna(-1).astype(int)

    requests_by_model = pdf.groupby("model_name_version").size().sort_values(ascending=False)
    status_counts = pdf.groupby("status_code").size().sort_index()
    requests_per_minute_by_model = (
        pdf.set_index(timestamp_col)
        .groupby(model_col)
        .resample("1min")
        .size()
        .unstack(level=0)
        .fillna(0)
    )
    status_by_model = (
        pdf.groupby([model_col, "status_code"])
        .size()
        .unstack(fill_value=0)
    )
    requester_share = pdf.groupby("requester").size().sort_values(ascending=False).head(5)
    summary_df = (
        pdf.groupby(model_col)
        .agg(total_requests=("databricks_request_id", "count"), unique_requesters=("requester", "nunique"))
        .sort_values("total_requests", ascending=False)
        .reset_index()
    )
    display(summary_df)

    model_palette = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
    status_palette = ["#16a34a", "#f59e0b", "#dc2626", "#6b7280", "#2563eb"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    requests_by_model.plot(kind="bar", ax=axes[0, 0], title="Requests by model/version", color=model_palette[:len(requests_by_model)])
    axes[0, 0].set_ylabel("requests")
    axes[0, 0].tick_params(axis="x", rotation=20)

    status_counts.plot(kind="bar", ax=axes[0, 1], title="Requests by status code", color=status_palette[:len(status_counts)])
    axes[0, 1].set_ylabel("requests")
    axes[0, 1].tick_params(axis="x", rotation=0)

    requests_per_minute_by_model.plot(ax=axes[1, 0], title="Requests per minute by model", color=model_palette[:len(requests_per_minute_by_model.columns)], linewidth=2)
    axes[1, 0].set_ylabel("requests / min")
    axes[1, 0].grid(axis="y", alpha=0.25)
    axes[1, 0].legend(title="model", loc="upper left")

    status_by_model.plot(kind="bar", stacked=True, ax=axes[1, 1], title="Status mix by model", color=status_palette[:len(status_by_model.columns)])
    axes[1, 1].set_ylabel("requests")
    axes[1, 1].tick_params(axis="x", rotation=20)
    axes[1, 1].legend(title="status", loc="upper right")

    plt.tight_layout()
    plt.show()

    if not requester_share.empty:
        plt.figure(figsize=(10, 4))
        requester_share.plot(kind="bar", color="#0f766e", title="Top requesters")
        plt.ylabel("requests")
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()

    latest_window = requests_per_minute_by_model.tail(10)
    display(latest_window.reset_index())